# 01c — Replogle 2022: Differential Expression

**Dataset:** ReplogleWeissman2022_rpe1 (scPerturb-harmonized, post-QC from 01a)  
**Accession:** GSE181897  
**Phase:** 2  
**Date:** 2026-03-11  
**Objective:**
1. Pseudobulk DE with PyDESeq2 for each perturbation vs. non-targeting controls
2. Secondary confirmation with Wilcoxon rank-sum (`sc.tl.rank_genes_groups`)
3. Perturbation similarity matrix from DE profiles + leiden clustering
4. Validate results on well-characterised essential genes

## Table of Contents

1. [Setup](#setup)
2. [Load Data](#load)
3. [Background: DE in Perturb-seq](#background)
   - 3a. Why differential expression?
   - 3b. The pseudobulk approach and why it is preferred
   - 3c. The negative binomial model (DESeq2 / PyDESeq2)
   - 3d. Wilcoxon rank-sum as a secondary check
4. [Pseudobulk Aggregation](#pseudobulk-agg) — aggregate raw counts per perturbation
5. [PyDESeq2: Single Perturbation Example](#pydeseq2-single) — detailed walk-through
6. [PyDESeq2: Screen-wide DE](#pydeseq2-screen) — all perturbations vs. control
7. [Wilcoxon DE (secondary)](#wilcoxon) — fast rank-based test for comparison
8. [Method Comparison: Pseudobulk vs. Wilcoxon](#method-compare)
9. [Perturbation Similarity Matrix](#similarity) — cosine distance on DE profiles
10. [Leiden Clustering of Perturbation Programs](#leiden-de)
11. [Biological Validation](#validation) — known essential genes
12. [Save Results](#save)
13. [Summary](#summary)

<a id='setup'></a>
## 0. Setup

In [ ]:
import anndata as ad
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import scipy.sparse as sp
from scipy.spatial.distance import pdist, squareform
import boto3
import warnings
warnings.filterwarnings('ignore')

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False)
np.random.seed(42)

from pathlib import Path
PROC_DIR  = Path('../../data/processed/scperturb')
RESULTS   = Path('../../results')
FIG_DIR   = RESULTS / 'figures'
TBL_DIR   = RESULTS / 'tables'
for d in [PROC_DIR, FIG_DIR, TBL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

S3_BUCKET = 'learn-perturb-seq'

# Use the scVI-processed file from 01b (has all embeddings),
# or fall back to QC-only file from 01a
SCVI_PATH = PROC_DIR / 'ReplogleWeissman2022_rpe1_scvi.h5ad'
QC_PATH   = PROC_DIR / 'ReplogleWeissman2022_rpe1_qc.h5ad'

# DE thresholds
LFC_THRESHOLD  = 0.5     # minimum |log2 fold change| to call significant
FDR_THRESHOLD  = 0.05    # Benjamini-Hochberg adjusted p-value cutoff
MIN_CELLS_DE   = 20      # minimum cells per perturbation for DE

print('Setup complete.')
print(f'PyDESeq2 imported.')

<a id='load'></a>
## 1. Load Data

We need two representations of the data:
- **Raw counts** (`adata.raw`) for pseudobulk DE — count-based models require untransformed integers
- **Log-normalised expression** (`adata.X`) for Wilcoxon rank-sum and visualisation

In [ ]:
# Load processed AnnData
load_path = SCVI_PATH if SCVI_PATH.exists() else QC_PATH
if not load_path.exists():
    # Try S3
    s3 = boto3.client('s3')
    for candidate in [SCVI_PATH, QC_PATH]:
        try:
            s3.download_file(S3_BUCKET, f'processed/scperturb/{candidate.name}', str(candidate))
            load_path = candidate
            print(f'Downloaded {candidate.name} from S3.')
            break
        except Exception:
            continue
    else:
        raise FileNotFoundError('Run 01a/01b first to produce the processed AnnData.')

adata = sc.read_h5ad(load_path)
print(f'Loaded: {load_path.name}')
print(f'  Cells: {adata.n_obs:,}  Genes: {adata.n_vars:,}')
print(f'  Raw counts available: {adata.raw is not None}')

# Recover raw count AnnData for pseudobulk DE
if adata.raw is not None:
    adata_raw = adata.raw.to_adata()
    print(f'  Raw AnnData: {adata_raw.shape}')
else:
    raise ValueError('adata.raw is required for pseudobulk DE. Re-run 01a.')

# Filter to cells with clean guide assignment and sufficient cells per perturbation
mask_clean = ~adata.obs['guide_ambiguous'] if 'guide_ambiguous' in adata.obs.columns else pd.Series(True, index=adata.obs_names)
mask_single = adata.obs['nperts'] == 1

print(f'\nFiltering to single-perturbation, clean-guide cells:')
print(f'  nperts == 1:      {mask_single.sum():,}')
print(f'  guide_ambiguous == False: {mask_clean.sum():,}')
print(f'  Both:             {(mask_single & mask_clean).sum():,}')

adata     = adata[mask_single & mask_clean].copy()
adata_raw = adata_raw[adata.obs_names].copy()

# Identify perturbations with enough cells
pert_counts = adata.obs['perturbation'].value_counts()
perts_for_de = pert_counts[pert_counts >= MIN_CELLS_DE].index.tolist()
if 'control' in perts_for_de:
    perts_for_de.remove('control')
n_ctrl = (adata.obs['perturbation'] == 'control').sum()

print(f'\nPerturbations with >= {MIN_CELLS_DE} cells: {len(perts_for_de):,}')
print(f'Control cells: {n_ctrl:,}')

<a id='background'></a>
## 2. Background: Differential Expression in Perturb-seq

### 3a. Why differential expression?

The fundamental question in a Perturb-seq experiment is: **for each gene knocked down,
which other genes change expression as a result?** This is the perturbation's
*transcriptional signature* — the set of downstream genes that are up- or down-regulated.

The DE signature is the foundation for:
- **Gene function annotation:** What pathways does this gene regulate?
- **Perturbation similarity:** Which perturbations produce overlapping signatures
  (implying shared pathways)?
- **GWAS cross-reference:** Does a GWAS-nominated gene's perturbation signature
  overlap with known disease biology?

---

### 3b. The pseudobulk approach — and why it is preferred

#### The independence problem

In single-cell DE, the most common mistake is treating each cell as an independent
observation. Consider a perturbation group with 500 cells:

- **Wilcoxon / t-test perspective:** 500 independent replicates → enormous statistical
  power → everything is significant → inflated Type I error
- **Reality:** All 500 cells were derived from the same pool, infected with the same
  guide RNA library, processed in the same lane, sequenced in the same run. They are
  **technical replicates**, not biological replicates.

The consequence: naive single-cell tests (Wilcoxon, t-test) produce **p-values that are
orders of magnitude too small**, leading to thousands of false-positive DE genes.

#### What pseudobulk does

Pseudobulk aggregation collapses the single-cell resolution to the perturbation level:

```
Perturbation A (500 cells):          Pseudobulk sample A:
  Cell 1:  [3, 0, 12, 5, ...]            Sum:  [1850, 420, 5640, 2100, ...]
  Cell 2:  [5, 2,  8, 3, ...]
  ...                                    → one "sample" per perturbation
  Cell 500: [2, 0, 15, 7, ...]
```

Each perturbation becomes a single "sample" with one count vector (the sum of all
cells' raw counts). This restores the statistical framework to the sample level,
where the unit of replication is the perturbation group, not the individual cell.

#### The replication challenge

DESeq2 / PyDESeq2 requires **multiple samples** per condition to estimate within-group
variance. In a typical Perturb-seq experiment, each perturbation appears only once
(one guide per gene), so we cannot estimate perturbation-specific variance.

**Workarounds:**

1. **Treat each perturbation as a replicate of the "perturbed" condition** and test
   against the control condition (which has many pseudo-replicates from subsampling).
   This is a *per-perturbation test with shared variance estimation* — DESeq2 borrows
   strength across perturbations to estimate dispersion.

2. **Split control cells into pseudo-replicates** (e.g., random thirds) to provide
   within-condition variance estimates for the control group.

3. **Use the Wald test with shrunken LFC** (DESeq2's default), which is more robust
   to low replicate counts than the likelihood ratio test.

We use approach (2): split control cells into `k` pseudo-replicates, and treat each
perturbation + its `k` control pseudo-replicates as a 1-vs-k comparison.

---

### 3c. The negative binomial model (DESeq2 / PyDESeq2)

PyDESeq2 models each gene's counts with a **negative binomial (NB) distribution**:

$$K_{ij} \sim \text{NB}(\mu_{ij}, \alpha_i)$$

where:
- $K_{ij}$ = raw count of gene $i$ in sample $j$ (the pseudobulk count)
- $\mu_{ij} = s_j \cdot q_{ij}$ = expected count = size factor × normalised mean
- $s_j$ = size factor for sample $j$ (accounts for library size differences)
- $q_{ij}$ = true expression level of gene $i$ in condition $j$
- $\alpha_i$ = dispersion parameter for gene $i$ (biological variability)

**The variance of the NB model:**

$$\text{Var}(K_{ij}) = \mu_{ij} + \alpha_i \cdot \mu_{ij}^2$$

The first term ($\mu_{ij}$) is Poisson sampling noise. The second term
($\alpha_i \cdot \mu_{ij}^2$) captures biological overdispersion — the extra
variability that cannot be explained by random sampling alone.

**The three-step DESeq2 pipeline:**

1. **Size factor estimation** (`estimateSizeFactors`): Compute a normalisation
   factor $s_j$ for each sample using the median-of-ratios method. This corrects
   for differences in total sequencing depth across pseudobulk samples.

2. **Dispersion estimation** (`estimateDispersions`): For each gene, estimate the
   dispersion $\alpha_i$. DESeq2 uses an empirical Bayes approach:
   - First, estimate a gene-wise dispersion from the data (noisy)
   - Then fit a smooth trend: $\alpha(\mu)$ — dispersion as a function of mean
     expression (genes with higher mean tend to have lower dispersion)
   - Finally, shrink each gene's dispersion toward the trend using a Bayesian prior
   - This "borrowing strength across genes" is crucial when sample sizes are small

3. **Wald test** (`nbinomWaldTest`): For each gene, test the null hypothesis
   $H_0: \log_2 FC = 0$ using a Wald statistic:
   $$W = \frac{\hat{\beta}}{\text{SE}(\hat{\beta})}$$
   where $\hat{\beta}$ is the estimated log2 fold change and SE is its standard error.
   The Wald statistic follows an approximate standard normal under $H_0$.

**Log2 fold change shrinkage (apeglm):**

DESeq2 can apply LFC shrinkage (apeglm method) to produce more reliable effect-size
estimates. Genes with low counts or high variability have their LFC shrunk toward zero,
reducing noise without affecting the p-value. This is especially useful for building
perturbation similarity matrices, where we want the LFC profile to be a reliable signal.

---

### 3d. Wilcoxon rank-sum as a secondary check

The Wilcoxon rank-sum test (Mann-Whitney U) is a non-parametric test that compares
the *ranks* of expression values between two groups:

1. Pool all cells from the perturbation and control groups
2. Rank all cells by expression of gene $g$
3. The test statistic = sum of ranks in the perturbation group
4. Under $H_0$ (no difference), the rank sum follows a known distribution → p-value

**Advantages:**
- Very fast (no model fitting)
- No distributional assumptions (works on ranks)
- Good for quick screening

**Limitations for Perturb-seq:**
- Treats cells as independent (inflates significance)
- Cannot estimate fold change directly (only ranks)
- Sensitive to library-size differences between groups

We use Wilcoxon as a **secondary confirmation**: if pseudobulk calls a gene as DE,
we expect Wilcoxon to agree (though with much smaller p-values). Genes called by
Wilcoxon but not pseudobulk are likely false positives from the independence violation.

<a id='pseudobulk-agg'></a>
## 3. Pseudobulk Aggregation

We aggregate raw counts per perturbation group by summing across all cells assigned
to each perturbation. Control cells are split into `N_CTRL_SPLITS` pseudo-replicates
(random, non-overlapping subsets) to provide within-condition variance estimates.

In [ ]:
# ---- Pseudobulk aggregation -----------------------------------------------
# For each perturbation: sum raw counts across all cells in that group.
# For controls: split into N_CTRL_SPLITS pseudo-replicates.

N_CTRL_SPLITS = 3

raw_X = adata_raw.X
if sp.issparse(raw_X):
    raw_X_csr = raw_X.tocsr()
else:
    raw_X_csr = sp.csr_matrix(raw_X)

gene_names = adata_raw.var_names.tolist()

# --- Aggregate perturbed groups ---
pb_counts  = {}   # sample_name -> count vector
pb_meta    = {}   # sample_name -> metadata dict

for pert in perts_for_de:
    idx = np.where(adata.obs['perturbation'].values == pert)[0]
    counts = np.array(raw_X_csr[idx].sum(axis=0)).flatten()
    sample_name = f'pert_{pert}'
    pb_counts[sample_name] = counts
    pb_meta[sample_name]   = {'condition': 'perturbed', 'perturbation': pert, 'n_cells': len(idx)}

# --- Split control cells into pseudo-replicates ---
ctrl_idx = np.where(adata.obs['perturbation'].values == 'control')[0]
rng = np.random.default_rng(42)
rng.shuffle(ctrl_idx)
ctrl_splits = np.array_split(ctrl_idx, N_CTRL_SPLITS)

for k, split_idx in enumerate(ctrl_splits):
    counts = np.array(raw_X_csr[split_idx].sum(axis=0)).flatten()
    sample_name = f'ctrl_rep{k}'
    pb_counts[sample_name] = counts
    pb_meta[sample_name]   = {'condition': 'control', 'perturbation': 'control', 'n_cells': len(split_idx)}

# Assemble into a DataFrame (samples x genes)
pb_df   = pd.DataFrame(pb_counts, index=gene_names).T
pb_obs  = pd.DataFrame(pb_meta).T

print(f'Pseudobulk count matrix: {pb_df.shape[0]} samples x {pb_df.shape[1]} genes')
print(f'  Perturbation samples: {len(perts_for_de)}')
print(f'  Control pseudo-replicates: {N_CTRL_SPLITS}')
print(f'\nCount range: [{pb_df.values.min():.0f}, {pb_df.values.max():.0f}]')
print(f'Sample library sizes (sum per sample):')
print(pb_df.sum(axis=1).describe().round(0))

<a id='pydeseq2-single'></a>
## 4. PyDESeq2: Single Perturbation Walk-through

Before running the full screen, we walk through the DESeq2 pipeline for a single
perturbation in detail. This lets us inspect the intermediate results and build
intuition for what the model is doing.

We pick a well-characterised perturbation with a strong expected effect.

In [ ]:
# Pick a well-characterised perturbation for the walk-through.
# Strategy: choose the perturbation with the most cells (highest statistical power)
# among the top tier.
top_pert = pert_counts[pert_counts.index.isin(perts_for_de)].head(10)
print('Top 10 perturbations by cell count (candidates for walk-through):')
for p, n in top_pert.items():
    print(f'  {p:20s}  {n:,} cells')

# Select the top one
TARGET = top_pert.index[0]
n_target = top_pert.iloc[0]
print(f'\nSelected: {TARGET} ({n_target:,} cells)')

In [ ]:
# ---- Build a small pseudobulk dataset: one perturbation + control reps ----
sample_names = [f'pert_{TARGET}'] + [f'ctrl_rep{k}' for k in range(N_CTRL_SPLITS)]
counts_sub  = pb_df.loc[sample_names].copy()
meta_sub    = pb_obs.loc[sample_names].copy()

# PyDESeq2 requires the design factor as a column in the metadata
print(f'Samples in this comparison:')
for name, row in meta_sub.iterrows():
    print(f'  {name:20s}  condition={row["condition"]:10s}  n_cells={row["n_cells"]}')

# ---- Step 1: Create the DeseqDataSet ------------------------------------
# The design_factors parameter specifies the experimental variable to test.
# Here, 'condition' has two levels: 'perturbed' and 'control'.
dds = DeseqDataSet(
    counts=counts_sub,
    metadata=meta_sub,
    design_factors='condition',
)
print(f'\nDeseqDataSet created: {dds.n_obs} samples, {dds.n_vars} genes')

# ---- Step 2: Run the DESeq2 pipeline ------------------------------------
# This performs: (1) size factor estimation, (2) dispersion estimation, (3) Wald test
dds.deseq2()
print('DESeq2 pipeline complete.')

# Inspect size factors
print(f'\nSize factors (library size normalisation):')
for name, sf in zip(dds.obs_names, dds.obsm['size_factors']):
    print(f'  {name:20s}  s = {sf:.4f}')

In [ ]:
# ---- Step 3: Extract results for the contrast perturbed vs. control ------
# The 'contrast' specifies: [factor, numerator, denominator]
# log2FC = log2(perturbed / control) → positive = up in perturbation
stat_res = DeseqStats(
    dds,
    contrast=['condition', 'perturbed', 'control'],
)
stat_res.summary()

# Extract the results DataFrame
results_df = stat_res.results_df.copy()
results_df = results_df.sort_values('padj')

print(f'\nDE results for {TARGET} vs. control:')
print(f'  Total genes tested: {len(results_df):,}')

# Apply significance thresholds
sig = (results_df['padj'] < FDR_THRESHOLD) & (results_df['log2FoldChange'].abs() > LFC_THRESHOLD)
n_up   = ((sig) & (results_df['log2FoldChange'] > 0)).sum()
n_down = ((sig) & (results_df['log2FoldChange'] < 0)).sum()
print(f'  Significant (FDR<{FDR_THRESHOLD}, |LFC|>{LFC_THRESHOLD}): {sig.sum()} genes')
print(f'    Up-regulated:   {n_up}')
print(f'    Down-regulated: {n_down}')

print(f'\nTop 10 up-regulated genes:')
print(results_df[results_df['log2FoldChange'] > 0].head(10)[
    ['baseMean', 'log2FoldChange', 'pvalue', 'padj']
].to_string())
print(f'\nTop 10 down-regulated genes:')
print(results_df[results_df['log2FoldChange'] < 0].head(10)[
    ['baseMean', 'log2FoldChange', 'pvalue', 'padj']
].to_string())

In [ ]:
# ---- Volcano plot for the single perturbation example ---------------------
df_v = results_df.copy()
df_v['padj'] = df_v['padj'].clip(lower=1e-300)
df_v['neg_log10_padj'] = -np.log10(df_v['padj'])

sig_mask = (df_v['padj'] < FDR_THRESHOLD) & (df_v['log2FoldChange'].abs() > LFC_THRESHOLD)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(
    df_v.loc[~sig_mask, 'log2FoldChange'],
    df_v.loc[~sig_mask, 'neg_log10_padj'],
    s=8, alpha=0.3, color='steelblue', label='not significant', rasterized=True,
)
ax.scatter(
    df_v.loc[sig_mask, 'log2FoldChange'],
    df_v.loc[sig_mask, 'neg_log10_padj'],
    s=12, alpha=0.7, color='tomato', label=f'FDR<{FDR_THRESHOLD}, |LFC|>{LFC_THRESHOLD}', zorder=3,
)

# Label top 10 genes by significance
top_genes = df_v.nlargest(10, 'neg_log10_padj')
for _, row in top_genes.iterrows():
    ax.annotate(
        row.name, xy=(row['log2FoldChange'], row['neg_log10_padj']),
        xytext=(4, 2), textcoords='offset points', fontsize=7,
    )

ax.axvline( LFC_THRESHOLD, color='gray', ls='--', lw=0.7)
ax.axvline(-LFC_THRESHOLD, color='gray', ls='--', lw=0.7)
ax.axhline(-np.log10(FDR_THRESHOLD), color='gray', ls='--', lw=0.7, label=f'FDR={FDR_THRESHOLD}')
ax.set_xlabel('log2 fold change (perturbed / control)')
ax.set_ylabel(r'$-\log_{10}$(adjusted p-value)')
ax.set_title(f'Pseudobulk DE: {TARGET} CRISPRi vs. control (PyDESeq2)\n'
             f'{sig.sum()} significant genes ({n_up} up, {n_down} down)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / f'01c_volcano_pseudobulk_{TARGET}.pdf', bbox_inches='tight')
plt.show()

<a id='pydeseq2-screen'></a>
## 5. PyDESeq2: Screen-wide DE

Now we run pseudobulk DE for **every perturbation** (with >= 20 cells) against the
control pseudo-replicates. The result is a matrix of log2 fold changes
(perturbations × genes) that forms the basis of the perturbation similarity analysis.

### Implementation strategy

For a genome-scale screen with thousands of perturbations, running a separate DESeq2
model for each perturbation is computationally tractable because each model is small
(1 perturbation sample + 3 control samples = 4 samples). The dispersion estimation
is the bottleneck (~2 seconds per model), so the total time is roughly
`n_perturbations × 2 seconds`.

**Alternative: single multi-sample model**

An alternative is to fit a single DESeq2 model with all perturbations as levels of
the `condition` factor. This shares dispersion estimation across all samples (better
estimates) but requires more memory and produces many contrasts. We use the
per-perturbation approach here for simplicity and parallelisability.

In [ ]:
# ---- Screen-wide pseudobulk DE --------------------------------------------
# For each perturbation, run a 1-vs-3 DESeq2 comparison.
# Store: LFC, padj, and number of significant genes.

ctrl_samples = [f'ctrl_rep{k}' for k in range(N_CTRL_SPLITS)]

all_lfc   = {}    # perturbation -> Series of log2FC per gene
all_padj  = {}    # perturbation -> Series of adjusted p-values
de_summary = []   # list of dicts for summary table

# Limit to a manageable number if the screen is very large
MAX_PERTS = min(len(perts_for_de), 500)   # cap for tutorial speed
perts_to_test = perts_for_de[:MAX_PERTS]

print(f'Running pseudobulk DE for {len(perts_to_test)} perturbations...')
print(f'(control pseudo-replicates: {N_CTRL_SPLITS})')
print()

failed = []
for i, pert in enumerate(perts_to_test):
    sample_names_i = [f'pert_{pert}'] + ctrl_samples
    counts_i = pb_df.loc[sample_names_i].copy()
    meta_i   = pb_obs.loc[sample_names_i].copy()

    try:
        dds_i = DeseqDataSet(counts=counts_i, metadata=meta_i, design_factors='condition')
        dds_i.deseq2()
        stat_i = DeseqStats(dds_i, contrast=['condition', 'perturbed', 'control'])
        stat_i.summary()
        res_i = stat_i.results_df

        lfc_series  = res_i['log2FoldChange'].rename(pert)
        padj_series = res_i['padj'].rename(pert)
        all_lfc[pert]  = lfc_series
        all_padj[pert] = padj_series

        sig_i = (res_i['padj'] < FDR_THRESHOLD) & (res_i['log2FoldChange'].abs() > LFC_THRESHOLD)
        de_summary.append({
            'perturbation': pert,
            'n_cells':      int(pb_obs.loc[f'pert_{pert}', 'n_cells']),
            'n_sig':        int(sig_i.sum()),
            'n_up':         int(((sig_i) & (res_i['log2FoldChange'] > 0)).sum()),
            'n_down':       int(((sig_i) & (res_i['log2FoldChange'] < 0)).sum()),
            'max_lfc':      float(res_i['log2FoldChange'].max()),
            'min_lfc':      float(res_i['log2FoldChange'].min()),
        })
    except Exception as e:
        failed.append((pert, str(e)))

    if (i + 1) % 50 == 0 or (i + 1) == len(perts_to_test):
        print(f'  [{i+1:4d}/{len(perts_to_test)}] completed')

print(f'\nDE completed: {len(all_lfc)} succeeded, {len(failed)} failed.')
if failed:
    print(f'  Failed perturbations (first 10): {[f[0] for f in failed[:10]]}')

# Assemble into DataFrames
lfc_matrix  = pd.DataFrame(all_lfc).T    # perturbations x genes
padj_matrix = pd.DataFrame(all_padj).T
de_summary_df = pd.DataFrame(de_summary)

print(f'\nLFC matrix: {lfc_matrix.shape}')
print(f'padj matrix: {padj_matrix.shape}')

In [ ]:
# ---- Screen-wide DE summary -----------------------------------------------
print('Screen-wide DE summary:')
print(de_summary_df.describe().round(1))
print()

# Distribution of number of significant genes per perturbation
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(de_summary_df['n_sig'], bins=50, color='steelblue', edgecolor='none')
axes[0].set_xlabel('Number of significant DE genes')
axes[0].set_ylabel('Number of perturbations')
axes[0].set_title('DE gene count per perturbation')
axes[0].axvline(de_summary_df['n_sig'].median(), color='red', ls='--', lw=1,
                label=f'median={de_summary_df["n_sig"].median():.0f}')
axes[0].legend(fontsize=8)

# n_sig vs n_cells — are more cells giving more DE genes?
axes[1].scatter(de_summary_df['n_cells'], de_summary_df['n_sig'],
                s=10, alpha=0.4, color='steelblue', rasterized=True)
axes[1].set_xlabel('Cells per perturbation')
axes[1].set_ylabel('Significant DE genes')
axes[1].set_title('Statistical power: cells vs. DE genes')

# Top perturbations by number of DE genes
top20 = de_summary_df.nlargest(20, 'n_sig')
axes[2].barh(range(20), top20['n_sig'].values[::-1], color='steelblue')
axes[2].set_yticks(range(20))
axes[2].set_yticklabels(top20['perturbation'].values[::-1], fontsize=8)
axes[2].set_xlabel('Number of significant DE genes')
axes[2].set_title('Top 20 perturbations by DE gene count')

plt.suptitle(f'Pseudobulk DE summary — {len(de_summary_df)} perturbations tested', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / '01c_de_screen_summary.pdf', bbox_inches='tight')
plt.show()

<a id='wilcoxon'></a>
## 6. Wilcoxon DE (secondary confirmation)

We run a Wilcoxon rank-sum test on the log-normalised expression for the same
perturbation walk-through example, then compare results to the pseudobulk approach.

The key expectation: **Wilcoxon will find many more significant genes** because it
treats each cell as an independent observation, dramatically inflating test power.
Genes called by both methods are high-confidence hits.

In [ ]:
# ---- Wilcoxon DE for the walk-through perturbation -------------------------
# Use the log-normalised expression (adata.X, not raw counts)
adata_wlcx = adata[adata.obs['perturbation'].isin([TARGET, 'control'])].copy()

sc.tl.rank_genes_groups(
    adata_wlcx,
    groupby='perturbation',
    groups=[TARGET],
    reference='control',
    method='wilcoxon',
    key_added='wilcoxon_de',
)

wlcx_df = sc.get.rank_genes_groups_df(adata_wlcx, group=TARGET, key='wilcoxon_de')
wlcx_df = wlcx_df.rename(columns={'names': 'gene', 'logfoldchanges': 'lfc_wlcx', 'pvals_adj': 'padj_wlcx'})
wlcx_df = wlcx_df.set_index('gene')

sig_wlcx = (wlcx_df['padj_wlcx'] < FDR_THRESHOLD) & (wlcx_df['lfc_wlcx'].abs() > LFC_THRESHOLD)
print(f'Wilcoxon DE for {TARGET}:')
print(f'  Significant genes: {sig_wlcx.sum()}')
print(f'  Up:   {((sig_wlcx) & (wlcx_df["lfc_wlcx"] > 0)).sum()}')
print(f'  Down: {((sig_wlcx) & (wlcx_df["lfc_wlcx"] < 0)).sum()}')

# Compare to pseudobulk
pb_sig_genes  = set(results_df.index[sig])
wlcx_sig_genes = set(wlcx_df.index[sig_wlcx])

both   = pb_sig_genes & wlcx_sig_genes
pb_only   = pb_sig_genes - wlcx_sig_genes
wlcx_only = wlcx_sig_genes - pb_sig_genes

print(f'\nMethod comparison for {TARGET}:')
print(f'  Pseudobulk only:       {len(pb_only):>5}')
print(f'  Wilcoxon only:         {len(wlcx_only):>5}')
print(f'  Both methods agree:    {len(both):>5}')
print(f'  Pseudobulk total:      {len(pb_sig_genes):>5}')
print(f'  Wilcoxon total:        {len(wlcx_sig_genes):>5}')

<a id='method-compare'></a>
## 7. Method Comparison: Pseudobulk vs. Wilcoxon

We compare the log2 fold changes and p-values between the two methods to understand
how the pseudobulk approach differs from the naive single-cell test.

**Expected patterns:**
- LFC estimates should be **correlated** between methods (they measure the same effect)
- Pseudobulk p-values should be **larger** (more conservative, fewer false positives)
- Wilcoxon will call many more genes significant (inflated by the independence assumption)
- High-confidence DE genes: those called by both methods

In [ ]:
# ---- LFC and p-value comparison plots -------------------------------------
# Merge pseudobulk and Wilcoxon results on gene name
compare = results_df[['log2FoldChange', 'padj']].copy()
compare.columns = ['lfc_pb', 'padj_pb']
compare = compare.join(wlcx_df[['lfc_wlcx', 'padj_wlcx']], how='inner')
compare = compare.dropna()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# LFC scatter
axes[0].scatter(compare['lfc_pb'], compare['lfc_wlcx'],
                s=5, alpha=0.3, color='steelblue', rasterized=True)
r_lfc = compare[['lfc_pb', 'lfc_wlcx']].corr().iloc[0, 1]
axes[0].plot([-5, 5], [-5, 5], 'r--', lw=0.8)
axes[0].set_xlabel('Pseudobulk log2FC')
axes[0].set_ylabel('Wilcoxon log2FC')
axes[0].set_title(f'LFC comparison (r = {r_lfc:.3f})')

# P-value scatter (log scale)
axes[1].scatter(
    -np.log10(compare['padj_pb'].clip(1e-300)),
    -np.log10(compare['padj_wlcx'].clip(1e-300)),
    s=5, alpha=0.3, color='steelblue', rasterized=True,
)
max_val = max(-np.log10(compare['padj_pb'].clip(1e-300)).max(),
              -np.log10(compare['padj_wlcx'].clip(1e-300)).max())
axes[1].plot([0, max_val], [0, max_val], 'r--', lw=0.8)
axes[1].set_xlabel(r'Pseudobulk $-\log_{10}$(padj)')
axes[1].set_ylabel(r'Wilcoxon $-\log_{10}$(padj)')
axes[1].set_title('P-value comparison\n(Wilcoxon more significant = above diagonal)')

# Venn-like bar chart
bar_data = [len(pb_only), len(both), len(wlcx_only)]
bar_labels = ['Pseudobulk\nonly', 'Both', 'Wilcoxon\nonly']
colors_bar = ['steelblue', 'purple', 'orange']
axes[2].bar(bar_labels, bar_data, color=colors_bar)
axes[2].set_ylabel('Number of significant genes')
axes[2].set_title(f'Method agreement for {TARGET}')
for j, (lbl, val) in enumerate(zip(bar_labels, bar_data)):
    axes[2].text(j, val + 2, str(val), ha='center', fontsize=10, fontweight='bold')

plt.suptitle(f'Pseudobulk vs. Wilcoxon DE — {TARGET}', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / f'01c_method_comparison_{TARGET}.pdf', bbox_inches='tight')
plt.show()

<a id='similarity'></a>
## 8. Perturbation Similarity Matrix

Each perturbation now has a **DE profile** — a vector of log2 fold changes across all
genes. We can compare these profiles to identify perturbations with similar downstream
effects, which implies they regulate the same biological pathways.

### Cosine similarity vs. Euclidean distance

We use **cosine distance** on the LFC profiles rather than Euclidean distance because:

- **Scale-invariant:** A perturbation with uniformly small LFCs (weak effect) and one with
  large LFCs (strong effect) in the same direction will have high cosine similarity.
  Euclidean distance would call them "far apart" simply because of the magnitude difference.
- **Robust to noise:** Genes with LFC ≈ 0 (no effect) contribute little to the cosine
  similarity, effectively down-weighting noise dimensions.
- **Standard in the field:** Cosine similarity on DE profiles is the most common approach
  for perturbation comparisons (Replogle 2022, Norman 2019).

**Cosine distance** = 1 − cosine similarity:

$$d_{\cos}(\mathbf{a}, \mathbf{b}) = 1 - \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \, \|\mathbf{b}\|}$$

- $d = 0$: identical LFC profiles (same direction, same affected genes)
- $d = 1$: orthogonal profiles (no shared DE genes)
- $d = 2$: anti-correlated profiles (opposite directions)

In [ ]:
# ---- Build the perturbation similarity matrix ------------------------------
# Use LFC values for all genes; fill NaN with 0 (no effect = zero LFC)
lfc_filled = lfc_matrix.fillna(0)

# Compute pairwise cosine distance
cosine_dist = squareform(pdist(lfc_filled.values, metric='cosine'))
cosine_sim  = 1 - cosine_dist

pert_names = lfc_filled.index.tolist()
sim_df = pd.DataFrame(cosine_sim, index=pert_names, columns=pert_names)

print(f'Similarity matrix: {sim_df.shape}')
print(f'Similarity range: [{cosine_sim.min():.3f}, {cosine_sim.max():.3f}]')
print(f'Mean pairwise similarity: {cosine_sim[np.triu_indices_from(cosine_sim, k=1)].mean():.3f}')

# ---- Heatmap of the top N perturbations by DE gene count -------------------
# Focus on perturbations with the strongest effects for visualisation
N_TOP = min(50, len(pert_names))
top_perts_de = de_summary_df.nlargest(N_TOP, 'n_sig')['perturbation'].tolist()
top_perts_de = [p for p in top_perts_de if p in sim_df.index]

sim_top = sim_df.loc[top_perts_de, top_perts_de]

fig, ax = plt.subplots(figsize=(12, 10))
sns.clustermap(
    sim_top,
    cmap='RdBu_r', center=0, vmin=-0.5, vmax=1.0,
    xticklabels=True, yticklabels=True,
    linewidths=0.2, figsize=(13, 11),
    dendrogram_ratio=0.12, cbar_pos=(0.02, 0.82, 0.03, 0.15),
    cbar_kws={'label': 'Cosine similarity'},
)
plt.suptitle(f'Perturbation similarity (cosine on LFC profiles)\nTop {len(top_perts_de)} perturbations by DE gene count',
             y=1.02, fontsize=12)
plt.savefig(FIG_DIR / '01c_perturbation_similarity_heatmap.pdf', bbox_inches='tight')
plt.show()

<a id='leiden-de'></a>
## 9. Leiden Clustering of Perturbation Programs

We cluster perturbations by their DE profiles to identify **shared transcriptional
programs** — groups of perturbations that produce similar downstream expression changes.
These clusters are biologically interpretable:

- **Pathway co-membership:** Genes in the same pathway (e.g., mitochondrial electron
  transport chain) will produce similar DE profiles when knocked down
- **Regulatory networks:** A transcription factor and its direct targets should cluster
  together (similar downstream effects)
- **Compensatory mechanisms:** Genes with opposite functions may anti-cluster (anticorrelated
  DE profiles)

We build a k-NN graph from the cosine similarity matrix and run Leiden clustering.

In [ ]:
# ---- Build an AnnData of perturbations for clustering ----------------------
# Each "cell" is a perturbation, each "gene" is an LFC value
adata_pert = ad.AnnData(
    X=lfc_filled.values,
    obs=pd.DataFrame(index=lfc_filled.index),
    var=pd.DataFrame(index=lfc_filled.columns),
)
adata_pert.obs = adata_pert.obs.join(de_summary_df.set_index('perturbation'), how='left')

# PCA on the LFC matrix (for k-NN graph)
N_PCA_PERT = min(30, lfc_filled.shape[0] - 1, lfc_filled.shape[1] - 1)
sc.tl.pca(adata_pert, n_comps=N_PCA_PERT)
sc.pp.neighbors(adata_pert, n_neighbors=15, use_rep='X_pca')

# Leiden clustering
sc.tl.leiden(adata_pert, resolution=0.8, key_added='leiden_de', random_state=42)
sc.tl.umap(adata_pert, random_state=42)

n_clusters = adata_pert.obs['leiden_de'].nunique()
print(f'Leiden clustering on DE profiles: {n_clusters} clusters')
print()
print('Perturbations per cluster:')
for cl in sorted(adata_pert.obs['leiden_de'].unique(), key=int):
    members = adata_pert.obs[adata_pert.obs['leiden_de'] == cl].index.tolist()
    print(f'  Cluster {cl}: {len(members):3d} perturbations')
    if len(members) <= 5:
        print(f'    Members: {members}')
    else:
        print(f'    Top 5 by DE genes: {adata_pert.obs.loc[members].nlargest(5, "n_sig").index.tolist()}')

In [ ]:
# ---- Visualise perturbation clusters on UMAP -------------------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: Leiden clusters
sc.pl.umap(adata_pert, color='leiden_de', ax=axes[0], show=False,
           title='Perturbation UMAP: Leiden clusters (from DE profiles)',
           legend_loc='right margin', s=30)

# Right: colour by number of significant DE genes
sc.pl.umap(adata_pert, color='n_sig', ax=axes[1], show=False,
           title='Perturbation UMAP: coloured by n_sig DE genes',
           color_map='viridis', s=30)

plt.tight_layout()
plt.savefig(FIG_DIR / '01c_perturbation_leiden_umap.pdf', bbox_inches='tight')
plt.show()

<a id='validation'></a>
## 10. Biological Validation

We validate the DE pipeline by checking whether known biology is recovered.

### Validation strategy

1. **Self-regulation check:** When gene X is knocked down by CRISPRi, gene X itself
   should be among the top down-regulated genes in its DE profile. This is a basic
   sanity check — if the knockdown worked, the targeted gene's expression should drop.

2. **Known pathway genes:** For well-characterised perturbations, check whether known
   downstream targets appear in the DE list.

3. **Essential gene signatures:** Essential genes (those required for cell survival)
   should have more DE genes overall (because knocking them down triggers widespread
   stress responses).

In [ ]:
# ---- Validation 1: Self-regulation check -----------------------------------
# For each perturbation, check if the knocked-down gene is in the LFC matrix
# and whether its LFC is negative (expression decreased).
self_reg = []
for pert in lfc_matrix.index:
    if pert in lfc_matrix.columns:    # gene must be among the tested genes
        lfc_val  = lfc_matrix.loc[pert, pert]
        padj_val = padj_matrix.loc[pert, pert] if pert in padj_matrix.columns else np.nan
        self_reg.append({
            'perturbation': pert,
            'self_lfc':     lfc_val,
            'self_padj':    padj_val,
            'is_down':      lfc_val < 0 if not np.isnan(lfc_val) else False,
            'is_sig':       (padj_val < FDR_THRESHOLD and lfc_val < -LFC_THRESHOLD)
                            if not np.isnan(padj_val) else False,
        })

self_reg_df = pd.DataFrame(self_reg)
n_testable  = len(self_reg_df)
n_is_down   = self_reg_df['is_down'].sum()
n_is_sig    = self_reg_df['is_sig'].sum()

print(f'Self-regulation check:')
print(f'  Perturbations where target gene is testable: {n_testable}')
print(f'  Target gene is down-regulated (LFC < 0):     {n_is_down}  ({n_is_down/n_testable:.1%})')
print(f'  Target gene is significantly down:            {n_is_sig}   ({n_is_sig/n_testable:.1%})')

# Plot distribution of self-LFC values
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(self_reg_df['self_lfc'].dropna(), bins=50, color='steelblue', edgecolor='none')
ax.axvline(0, color='black', lw=1)
ax.axvline(-LFC_THRESHOLD, color='red', ls='--', lw=1, label=f'LFC = -{LFC_THRESHOLD}')
ax.set_xlabel('Self-LFC (knocked-down gene expression change)')
ax.set_ylabel('Number of perturbations')
ax.set_title('Self-regulation check: CRISPRi target gene LFC\n'
             f'{n_is_down}/{n_testable} ({n_is_down/n_testable:.0%}) show negative LFC')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / '01c_self_regulation_check.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ---- Validation 2: Essential gene signatures are broader -------------------
# Hypothesis: perturbations of essential genes should produce more DE genes
# (because the cell is under severe stress, triggering widespread expression changes).
# We use cells-per-perturbation as a proxy for essentiality:
# genes with very few surviving cells are candidates for essentials.

if 'n_cells' in de_summary_df.columns and 'n_sig' in de_summary_df.columns:
    df_val = de_summary_df.copy()
    
    # Define quintiles of cell count
    df_val['cell_quintile'] = pd.qcut(df_val['n_cells'], q=5, labels=[
        'Q1 (fewest cells)', 'Q2', 'Q3', 'Q4', 'Q5 (most cells)'
    ])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Box plot: n_sig by cell-count quintile
    quintile_order = ['Q1 (fewest cells)', 'Q2', 'Q3', 'Q4', 'Q5 (most cells)']
    boxplot_data = [df_val.loc[df_val['cell_quintile'] == q, 'n_sig'].values for q in quintile_order]
    bp = axes[0].boxplot(boxplot_data, labels=[q.split('(')[0].strip() for q in quintile_order],
                         patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('steelblue')
        patch.set_alpha(0.6)
    axes[0].set_xlabel('Cell count quintile (Q1 = fewest surviving cells)')
    axes[0].set_ylabel('Number of significant DE genes')
    axes[0].set_title('Essential gene proxy: fewer cells → more DE genes?')
    
    # Scatter: n_cells vs n_sig, coloured by self-regulation status
    if len(self_reg_df) > 0:
        merged = df_val.merge(self_reg_df[['perturbation', 'is_sig']],
                              on='perturbation', how='left')
        merged['is_sig'] = merged['is_sig'].fillna(False)
        colors = merged['is_sig'].map({True: 'tomato', False: 'steelblue'})
        axes[1].scatter(merged['n_cells'], merged['n_sig'],
                       s=10, alpha=0.4, c=colors, rasterized=True)
        from matplotlib.patches import Patch
        axes[1].legend(handles=[
            Patch(color='tomato', label='Self-gene significantly down'),
            Patch(color='steelblue', label='Self-gene not sig. down'),
        ], fontsize=8)
    else:
        axes[1].scatter(df_val['n_cells'], df_val['n_sig'],
                       s=10, alpha=0.4, color='steelblue', rasterized=True)
    axes[1].set_xlabel('Cells per perturbation')
    axes[1].set_ylabel('Number of significant DE genes')
    axes[1].set_title('Cell count vs. DE breadth')
    
    plt.suptitle('Biological validation — essential gene signatures', y=1.02)
    plt.tight_layout()
    plt.savefig(FIG_DIR / '01c_essential_gene_validation.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# <a id='save'></a>
# === 11. Save results ======================================================

# Save LFC matrix (perturbations x genes)
lfc_path = TBL_DIR / '01c_lfc_matrix.csv.gz'
lfc_matrix.to_csv(lfc_path, compression='gzip')
print(f'LFC matrix saved to {lfc_path}')

# Save adjusted p-value matrix
padj_path = TBL_DIR / '01c_padj_matrix.csv.gz'
padj_matrix.to_csv(padj_path, compression='gzip')
print(f'padj matrix saved to {padj_path}')

# Save DE summary
de_summary_path = TBL_DIR / '01c_de_summary.csv'
de_summary_df.to_csv(de_summary_path, index=False)
print(f'DE summary saved to {de_summary_path}')

# Save perturbation similarity matrix
sim_path = TBL_DIR / '01c_cosine_similarity.csv.gz'
sim_df.to_csv(sim_path, compression='gzip')
print(f'Similarity matrix saved to {sim_path}')

# Save perturbation cluster assignments
cluster_path = TBL_DIR / '01c_perturbation_clusters.csv'
adata_pert.obs[['leiden_de', 'n_sig', 'n_up', 'n_down']].to_csv(cluster_path)
print(f'Cluster assignments saved to {cluster_path}')

# Save self-regulation check
self_reg_path = TBL_DIR / '01c_self_regulation_check.csv'
self_reg_df.to_csv(self_reg_path, index=False)
print(f'Self-regulation check saved to {self_reg_path}')

# Upload to S3
s3 = boto3.client('s3')
for fpath in [lfc_path, padj_path, de_summary_path, sim_path, cluster_path, self_reg_path]:
    try:
        s3.upload_file(str(fpath), S3_BUCKET, f'results/tables/{fpath.name}')
    except Exception:
        pass
print('\nS3 upload attempted for all result tables.')

In [ ]:
# <a id='summary'></a>
print('=' * 65)
print('01c DIFFERENTIAL EXPRESSION — SUMMARY')
print('=' * 65)
print()

checks = [
    ('Pseudobulk aggregation completed',                 pb_df.shape[0] > 0),
    (f'Pseudobulk DE run for walk-through ({TARGET})',   results_df is not None),
    (f'Screen-wide DE run ({len(all_lfc)} perturbations)',  len(all_lfc) > 0),
    ('Wilcoxon DE (secondary) run',                      wlcx_df is not None),
    ('Method comparison plotted',                        True),
    (f'Perturbation similarity matrix ({sim_df.shape})', sim_df.shape[0] > 0),
    (f'Leiden clustering ({n_clusters} clusters)',       n_clusters > 0),
    ('Self-regulation check completed',                  len(self_reg_df) > 0),
    ('Essential gene validation plotted',                True),
    ('LFC + padj matrices saved',                        lfc_path.exists()),
]

all_pass = True
for check, result in checks:
    status = 'v' if result else 'X'
    print(f'  [{status}]  {check}')
    if not result:
        all_pass = False

print()
print(f'  Walk-through perturbation: {TARGET}')
print(f'    Pseudobulk sig genes: {sig.sum()} ({n_up} up, {n_down} down)')
print(f'    Wilcoxon sig genes:   {sig_wlcx.sum()}')
print(f'    Both methods agree:   {len(both)}')
print(f'  Screen-wide:')
print(f'    Perturbations tested: {len(all_lfc)}')
print(f'    Median sig genes:     {de_summary_df["n_sig"].median():.0f}')
print(f'    Self-regulation pass: {n_is_down}/{n_testable} ({n_is_down/n_testable:.0%})')
print(f'    DE clusters:          {n_clusters}')
print()
print('Phase 2 DE checkpoint:', 'PASSED' if all_pass else 'INCOMPLETE (see above)')
print()
print('Next steps:')
print('  01d_signature_scoring.ipynb — GSEA / ORA on these DE gene lists')
print('  01e_cpa_gene_programs.ipynb — CPA model + CINEMA-OT gene programs')

In [ ]:
from datetime import datetime
import subprocess

timestamp   = datetime.now().strftime('%Y%m%d_%H%M')
report_dir  = Path('../../results/reports')
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / f'01c_differential_expression_{timestamp}.html'

nb_path = (
    __vsc_ipynb_file__
    if '__vsc_ipynb_file__' in dir()
    else '01c_differential_expression.ipynb'
)
subprocess.run(
    ['jupyter', 'nbconvert', '--to', 'html', '--output', str(report_path), nb_path],
    check=False,
)
print(f'Report saved to {report_path}')